# 🛠️ LangChain Tools — Full Demo
### Topics: Tool Creation · State · Context · Store · Streaming

In [ ]:
# Cell 1: Install dependencies
!pip install langchain langgraph langchain-openai python-dotenv

In [ ]:
# Cell 2: LLM Configuration
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)

print("✅ LLM configured!")

## 🟦 STEP 1 — Tool Creation
> Tools are just Python functions the LLM can call when needed.  
> `@tool` decorator makes any function usable by an agent.  
> `runtime` gives access to state, context, memory — hidden from LLM.

In [ ]:
# Cell 3: Basic Tool Creation
from langchain.tools import tool

@tool
def greet_user(name: str) -> str:
    """Greet a user by name."""
    return f"Hello, {name}! Welcome aboard."

# Test it directly
print(greet_user.invoke({"name": "Priya"}))
print("Tool name       :", greet_user.name)
print("Tool description:", greet_user.description)

## 🟩 STEP 2 — State (Short-term Memory)
> State lives for the duration of one conversation.  
> Tools update state using `Command`.  
> `runtime.tool_call_id` links the tool response back to the correct call.

In [ ]:
# Cell 4: Define State Tools
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

# Tool: Saves user name into state
@tool
def set_user_name(new_name: str, runtime: ToolRuntime) -> Command:
    """Remember the user's name for this conversation."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"Got it! I'll remember your name is {new_name}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

# Tool: Reads user name from state
@tool
def get_user_name(runtime: ToolRuntime) -> str:
    """Recall the user's name from memory."""
    name = runtime.state.get("user_name", None)
    if name:
        return f"Your name is {name}."
    return "I don't know your name yet. Please tell me!"

print("✅ State tools defined!")

In [ ]:
# Cell 5: Build Agent Graph with Custom State
from typing import Annotated, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages

# Custom state — adds user_name on top of messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    user_name: Optional[str]

tools_list = [set_user_name, get_user_name]
llm_with_tools = llm.bind_tools(tools_list)

def call_llm(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("llm", call_llm)
builder.add_node("tools", ToolNode(tools_list))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

state_graph = builder.compile()

print("✅ State agent graph compiled!")

In [ ]:
# Cell 6: DEMO — Short-term Memory in Action
from langchain_core.messages import HumanMessage

# STEP 1: Tell agent your name
result = state_graph.invoke({
    "messages": [HumanMessage(content="My name is Priya. Please remember it.")],
    "user_name": None
})

print("=== STEP 1: Set Name ===")
print("Agent :", result["messages"][-1].content)
print("State : user_name =", result["user_name"])   # ✅ Should show 'Priya'

# STEP 2: Ask agent to recall name (continue same conversation)
result2 = state_graph.invoke({
    "messages": result["messages"] + [HumanMessage(content="What is my name?")],
    "user_name": result["user_name"]
})

print("\n=== STEP 2: Recall Name ===")
print("Agent :", result2["messages"][-1].content)

## 🟨 STEP 3 — Context (Immutable User Data)
> Context = immutable config passed at invoke time (user ID, session info).  
> The LLM never sees it — but tools use it for personalization.  
> Same question → different answer based on who is asking.

In [ ]:
# Cell 7: Define Context Agent
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

USER_DATABASE = {
    "user123": {"name": "Alice Johnson", "account_type": "Premium",  "balance": 5000},
    "user456": {"name": "Bob Smith",     "account_type": "Standard", "balance": 1200},
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id
    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder : {user['name']}\n"
            f"Type           : {user['account_type']}\n"
            f"Balance        : ${user['balance']}"
        )
    return "User not found."

context_agent = create_agent(
    llm,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a helpful financial assistant."
)

print("✅ Context agent ready!")

In [ ]:
# Cell 8: DEMO — Same Question, Different Users
question = [{"role": "user", "content": "What's my current balance?"}]

# User 1 — Alice (Premium, $5000)
result_alice = context_agent.invoke(
    {"messages": question},
    context=UserContext(user_id="user123")
)
print("=== ALICE (user123) ===")
print(result_alice["messages"][-1].content)

# User 2 — Bob (Standard, $1200)
result_bob = context_agent.invoke(
    {"messages": question},
    context=UserContext(user_id="user456")
)
print("\n=== BOB (user456) ===")
print(result_bob["messages"][-1].content)

## 🟧 STEP 4 — Store (Long-term Memory)
> Store = persistent memory that survives across sessions.  
> Unlike state (resets every run), store data is always there.  
> Uses namespace + key pattern: `("users",)` → `"abc123"`

In [ ]:
# Cell 9: Define Store Agent
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user information to long-term memory."""
    runtime.store.put(("users",), user_id, user_info)
    return f"✅ Saved info for user '{user_id}'."

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Retrieve user information from long-term memory."""
    result = runtime.store.get(("users",), user_id)
    return str(result.value) if result else f"No info found for user '{user_id}'."

store = InMemoryStore()

memory_agent = create_agent(
    llm,
    tools=[save_user_info, get_user_info],
    store=store
)

print("✅ Memory agent with store ready!")

In [ ]:
# Cell 10: DEMO — Cross-Session Memory
# Messages list is completely fresh in Session 2 — simulates new session.
# But store still remembers. That's true persistence.

# SESSION 1 — Save user
print("=== SESSION 1: Saving user... ===")
session1 = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Save this user — id: abc123, name: Priya, age: 25, email: priya@demo.com"
    }]
})
print(session1["messages"][-1].content)

# SESSION 2 — Fresh messages, same store
print("\n=== SESSION 2: New session, retrieving user... ===")
session2 = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Get user info for user id 'abc123'"
    }]
})
print(session2["messages"][-1].content)

## 🟥 STEP 5 — Streaming (Real-time Updates)
> Streaming = show progress LIVE while the tool runs.  
> Uses `runtime.stream_writer` to emit updates during execution.  
> Critical for long-running operations like API calls or data processing.

In [ ]:
# Cell 11: Define Streaming Tool
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

@tool
def analyze_data(dataset_name: str, runtime: ToolRuntime) -> str:
    """Analyze a dataset and return insights."""
    writer = runtime.stream_writer

    writer(f"🔍 Fetching dataset  : {dataset_name}...")
    writer(f"📊 Running analysis  : {dataset_name}...")
    writer(f"✅ Analysis complete !")

    return f"Dataset '{dataset_name}': 1,200 records, avg value 42.5, 3 anomalies found."

stream_agent = create_agent(
    llm,
    tools=[analyze_data],
    system_prompt="You are a data analysis assistant."
)

print("✅ Streaming agent ready!")

In [ ]:
# Cell 12: DEMO — Live Streaming Updates
print("=== STREAMING DEMO ===\n")

for chunk in stream_agent.stream(
    {"messages": [{"role": "user", "content": "Analyze the sales_data dataset"}]},
    stream_mode="custom"
):
    print("📡 Live update:", chunk)

print("\n✅ Stream complete.")

## 🏆 Summary

In [ ]:
# Cell 13: Full Summary
summary = """
╔══════════════════════════════════════════════════════════════╗
║           LANGCHAIN TOOLS — FULL DEMO SUMMARY               ║
╠══════════════════════════════════════════════════════════════╣
║  🟦 Tool Creation  → @tool decorator, type hints            ║
║  🟩 State          → Short-term memory via Command          ║
║  🟨 Context        → Immutable user/session config          ║
║  🟧 Store          → Long-term memory across sessions       ║
║  🟥 Streaming      → Real-time updates via stream_writer    ║
╠══════════════════════════════════════════════════════════════╣
║  All accessed via ToolRuntime:                              ║
║    runtime.state         → current conversation state      ║
║    runtime.context       → user/session identity           ║
║    runtime.store         → persistent memory               ║
║    runtime.stream_writer → live progress updates           ║
║    runtime.tool_call_id  → execution tracking              ║
╠══════════════════════════════════════════════════════════════╣
║  LLM Config used throughout:                                ║
║    ChatOpenAI(model, base_url, api_key, temperature=0)      ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)